# Singer-Clark 2025 Study

`Singer2025CursorControl` standardizes each source block as one timeline. This notebook checks the three task families, materializes every continuous event type, and segments imagined cursor-control trials.

In [1]:
from neuralset.events.etypes import Event

from neuralshed.datasets.singer_2025.study import Singer2025CursorControl
from neuralshed.datasets.utils import dataset_dir
from neuralshed.extractors import BehavioralSeriesExtractor
from neuralshed.extractors import NeuralFeatureExtractor

## Representative timelines

Version 5 orders six day-39 Radial8 blocks first, then day-202 blocks, then day-468 blocks. Indices 0, 8, and 36 select one Radial8, simultaneous-speech, and grid-evaluation block.

In [2]:
study = Singer2025CursorControl(
    path=dataset_dir("singer_2025") / "data",
    query="timeline_index in [0, 8, 36]",
)
events = study.run()
print(events.groupby(["task", "type"]).size().to_string())

task                            type                
grid_evaluation                 Action                  70
                                BandPower                1
                                ClickAssistSeries        1
                                ClickStateSeries         1
                                CursorAssistSeries       1
                                CursorPositionSeries     1
                                DecodedCursorVelocitySeries     1
                                SpikeCounts              1
                                Stimulus                70
                                TargetPositionSeries     1
radial8_calibration             Action                  52
                                BandPower                1
                                CursorAssistSeries       1
                                CursorPositionSeries     1
                                DecodedCursorVelocitySeries     1
                                SpikeCounts     

In [3]:
columns = [
    "type",
    "start",
    "duration",
    "session",
    "run",
    "task",
    "original_split",
    "speech_condition",
]
events[columns].head(20)

,type,start,duration,session,run,task,original_split,speech_condition
0,SpikeCounts,0.00,179.86,t15_day00039,block00,radial8_calibration,calibration,none
1,BandPower,0.00,179.86,t15_day00039,block00,radial8_calibration,calibration,none
2,CursorPositionSeries,0.00,179.86,t15_day00039,block00,radial8_calibration,calibration,none
3,TargetPositionSeries,0.00,179.86,t15_day00039,block00,radial8_calibration,calibration,none
4,DecodedCursorVelocitySeries,0.00,179.86,t15_day00039,block00,radial8_calibration,calibration,none
5,CursorAssistSeries,0.00,179.86,t15_day00039,block00,radial8_calibration,calibration,none
6,Stimulus,0.00,2.51,t15_day00039,block00,radial8_calibration,calibration,none
7,Action,0.00,2.51,t15_day00039,block00,radial8_calibration,calibration,none
8,Stimulus,2.51,2.50,t15_day00039,block00,radial8_calibration,calibration,none
9,Action,2.51,2.50,t15_day00039,block00,radial8_calibration,calibration,none


## Lazy signal reads

Each source time-first array is exposed dims-first. Optional click streams appear only in click-enabled blocks.

In [4]:
stream_types = {
    "SpikeCounts",
    "BandPower",
    "CursorPositionSeries",
    "TargetPositionSeries",
    "DecodedCursorVelocitySeries",
    "CursorAssistSeries",
    "ClickAssistSeries",
    "ClickStateSeries",
}
for type_name in sorted(stream_types):
    rows = events[events.type == type_name]
    if rows.empty:
        continue
    array = Event.from_dict(rows.iloc[0].to_dict()).read()
    print(type_name, array.shape, array.dtype)

BandPower (256, 17986) float32
ClickAssistSeries (1, 18008) float32
ClickStateSeries (1, 18008) float32
CursorAssistSeries (1, 17986) float32
CursorPositionSeries (2, 17986) float32
DecodedCursorVelocitySeries (2, 17986) float32
SpikeCounts (256, 17986) float32
TargetPositionSeries (2, 17986) float32


## Trial segmentation

The `Action` rows with `action_kind == 'cursor_control'` define variable-duration movement trials. Neural inputs concatenate threshold crossings and spike-band power. The target is the online cursor velocity, kept separate from measured cursor position.

In [5]:
import neuralset as ns

segmenter = ns.dataloader.Segmenter(
    extractors={
        "neural": NeuralFeatureExtractor(
            target_unit="native",
            frequency=100,
            event_types=("SpikeCounts", "BandPower"),
            aggregation="cat",
        ),
        "velocity": BehavioralSeriesExtractor(
            frequency=100,
            event_types="DecodedCursorVelocitySeries",
        ),
    },
    trigger_query="type == 'Action' and action_kind == 'cursor_control'",
    duration=None,
)
segments = segmenter.apply(events)
segments.prepare()
segments.pad_duration = "auto"
sample = segments[0]
print("segments:", len(segments))
print({key: tuple(value.shape) for key, value in sample.data.items()})

2026-07-11 13:16:55 - INFO - neuralset.dataloader:222 - Preparing extractor: NeuralFeatureExtractor


  0%|          | 0/6 [00:00<?, ?it/s]

100%|██████████| 6/6 [00:00<00:00, 201.29it/s]


2026-07-11 13:16:55 - INFO - neuralset.dataloader:222 - Preparing extractor: BehavioralSeriesExtractor


  0%|          | 0/3 [00:00<?, ?it/s]

100%|██████████| 3/3 [00:00<00:00, 278.63it/s]

segments: 149
{'neural': (1, 512, 646), 'velocity': (1, 2, 646)}


The velocity target is itself produced online from the neural features. It is suitable for reproducing the deployed decoder output, but it must not be described as an independent measurement of intended movement. Calibration blocks should additionally be filtered by `CursorAssistSeries` when fully closed-loop behavior is required.